# Блокнот 1. Корпус, шрифты и генератор синтетических данных

Статья: «Синтез обучающих данных для нейросетевого распознавания печатного текста на татарском языке».

Что делает блокнот:
1. собирает татарский текстовый корпус (Википедия + книги с Hugging Face) и нормализует его;
2. находит шрифты, в которых есть все буквы татарского алфавита;
3. генерирует синтетические изображения строк в двух режимах: `clean` (без искажений) и `scan` (с искажениями сканирования);
4. верстает PDF для реального теста (распечатать → отсканировать).

Всё сохраняется на Google Диск в папку `tatar_ocr`, поэтому блокнот можно перезапускать: готовые шаги пропускаются.
GPU здесь не нужен (Среда выполнения → Сменить среду → CPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/tatar_ocr'      # всё сохраняется сюда
LOCAL = '/content/data'                        # быстрый локальный диск Colab
for d in [ROOT, f'{ROOT}/data', f'{ROOT}/print_test', LOCAL]:
    os.makedirs(d, exist_ok=True)
print('Готово:', ROOT)

## 1. Установка шрифтов и библиотек

In [ ]:
%%bash
# Пакеты шрифтов ставим по одному, чтобы отсутствие одного не ломало установку остальных
apt-get -qq update
for p in fonts-dejavu-core fonts-dejavu-extra fonts-liberation fonts-liberation2 \
         fonts-freefont-ttf fonts-noto-core fonts-crosextra-carlito fonts-paratype \
         fonts-ubuntu fonts-ibm-plex fonts-open-sans fonts-roboto; do
  apt-get -qq install -y $p > /dev/null 2>&1 && echo "ok  $p" || echo "--  $p (нет в репозитории, пропускаю)"
done
pip -q install datasets fonttools

## 2. Модуль генератора `tatsynth.py`

Алфавит модели, нормализация текста, проверка шрифтов, рендеринг строки, моделирование искажений сканирования, массовая генерация и вёрстка страниц для печати.
Искажения реализованы на OpenCV/NumPy без дополнительных библиотек.

In [ ]:
%%writefile tatsynth.py
# -*- coding: utf-8 -*-
"""tatsynth: синтез изображений строк печатного татарского текста.

Модуль содержит:
  * алфавит модели и нормализацию татарского текста;
  * нарезку корпуса на строки;
  * проверку покрытия шрифтов;
  * рендеринг строки и моделирование искажений, возникающих при сканировании.
Зависимости: numpy, Pillow, opencv-python, fontTools.
"""
import random, re, unicodedata
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont

# ---------------------------------------------------------------- алфавит
RU_LOWER = "абвгдеёжзийклмнопрстуфхцчшщъыьэюя"
TT_LOWER = "әөүҗңһ"            # U+04D9 U+04E9 U+04AF U+0497 U+04A3 U+04BB
TT_SPECIAL = set(TT_LOWER + TT_LOWER.upper())
LETTERS = RU_LOWER + TT_LOWER + (RU_LOWER + TT_LOWER).upper()
DIGITS = "0123456789"
PUNCT = " .,:;!?-–—()«»\"'%/№"
ALPHABET = LETTERS + DIGITS + PUNCT
ALPHABET_SET = set(ALPHABET)

# Похожие символы из других алфавитов -> правильные татарские буквы.
# Эти замены делаются всегда.
_ALWAYS = {
    "ə": "ә", "Ə": "Ә",   # латинская шва ə Ə
    "ӕ": "ә", "Ӕ": "Ә",   # кириллическая лигатура ӕ Ӕ (встречается по ошибке)
    "ɵ": "ө", "Ɵ": "Ө",   # латинская ɵ Ɵ
    "ѳ": "ө", "Ѳ": "Ө",   # фита ѳ Ѳ
    "ӂ": "җ", "Ӂ": "Җ",   # ж с краткой ӂ Ӂ
    "ҥ": "ң", "Ҥ": "Ң",   # алтайская ҥ Ҥ
    "ӈ": "ң", "Ӈ": "Ң",   # ӈ Ӈ
    "ӊ": "ң", "Ӊ": "Ң",   # ӊ Ӊ
    "ұ": "ү", "Ұ": "Ү",   # казахская ұ Ұ
    "ӓ": "ә", "Ӓ": "Ә",   # а с умлаутом
    "ӧ": "ө", "Ӧ": "Ө",   # о с умлаутом
    "ӱ": "ү", "Ӱ": "Ү",   # у с умлаутом
}
# Башкирские ҙ, ҫ и т. п. — самостоятельные буквы, их не трогаем:
# строки с ними просто отсеиваются фильтром is_clean.

# Латинские двойники кириллических букв: заменяем только внутри кириллических слов
_LAT2CYR = {
    "a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х", "y": "у",
    "h": "һ", "k": "к", "m": "м", "t": "т",
    "\u00FC": "ү", "\u00F6": "ө", "\u00E4": "ә",       # ü ö ä
    "A": "А", "E": "Е", "O": "О", "P": "Р", "C": "С", "X": "Х", "Y": "Ү",
    "H": "Һ", "K": "К", "M": "М", "T": "Т", "B": "В",
    "\u00DC": "Ү", "\u00D6": "Ө", "\u00C4": "Ә",
}
# Латинские h/H чаще всего набирают вместо һ/Һ (на русской раскладке их нет),
# а Y — вместо Ү, поэтому они отображаются именно в татарские буквы.
_PUNCT_MAP = {
    "‘": "'", "’": "'", "‚": "'", "ʼ": "'",
    "“": "\"", "”": "\"", "„": "«", "‟": "\"",
    "‒": "–", "―": "—", "−": "-", "‐": "-", "‑": "-",
    " ": " ", " ": " ", " ": " ", " ": " ",
    "…": "...",
}
_CYR = re.compile(r"[Ѐ-ӿ]")
_WORD = re.compile(r"[\wЀ-ӿ]+")


def _fix_word(w):
    """Латинские двойники заменяются, если в слове есть кириллица."""
    if not _CYR.search(w):
        return w
    return "".join(_LAT2CYR.get(ch, ch) for ch in w)


def normalize(text):
    text = unicodedata.normalize("NFC", text)
    text = "".join(_ALWAYS.get(ch, ch) for ch in text)
    text = "".join(_PUNCT_MAP.get(ch, ch) for ch in text)
    text = _WORD.sub(lambda m: _fix_word(m.group(0)), text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_clean(line, min_cyr_share=0.6):
    """Строка годится: все символы из алфавита, преобладает кириллица."""
    if not line or any(ch not in ALPHABET_SET for ch in line):
        return False
    letters = [ch for ch in line if ch.isalpha()]
    return len(letters) >= 3 and len(letters) / len(line) >= min_cyr_share * 0.7


def looks_tatar(text):
    """Грубая проверка, что текст татарский, а не русский."""
    n = sum(ch in TT_SPECIAL for ch in text)
    return n / max(1, len(text)) > 0.01


# ---------------------------------------------------------------- строки
_SENT = re.compile(r"(?<=[.!?…])\s+")


def text_to_lines(text, min_len=4, max_len=48, rng=random):
    """Нарезка нормализованного текста на строки длиной min_len..max_len."""
    out = []
    for sent in _SENT.split(text):
        words = sent.split(" ")
        i = 0
        while i < len(words):
            target = rng.randint(min_len, max_len)
            cur = words[i]
            i += 1
            while i < len(words) and len(cur) + 1 + len(words[i]) <= target:
                cur += " " + words[i]
                i += 1
            if min_len <= len(cur) <= max_len and is_clean(cur):
                out.append(cur)
    return out


def tatar_weight(line, boost=4.0):
    """Вес строки при выборке: строки с ә, ө, ү, җ, ң, һ берутся чаще."""
    return 1.0 + boost * min(3, sum(ch in TT_SPECIAL for ch in line))


# ---------------------------------------------------------------- шрифты
def font_covers(path, chars=LETTERS + DIGITS + PUNCT.replace(" ", "")):
    """Возвращает (покрывает_ли_всё, список_отсутствующих_символов)."""
    from fontTools.ttLib import TTFont, TTCollection
    try:
        if path.lower().endswith(".ttc"):
            fonts = TTCollection(path).fonts[:1]
        else:
            fonts = [TTFont(path, lazy=True)]
        cmap = fonts[0].getBestCmap() or {}
    except Exception:
        return False, list(chars)
    missing = [ch for ch in chars if ord(ch) not in cmap]
    return len(missing) == 0, missing


def glyphs_distinct(path, pairs=(("ә", "е"), ("ө", "о"), ("ү", "у"),
                                  ("җ", "ж"), ("ң", "н"), ("һ", None))):
    """Дополнительная проверка: татарская буква рисуется не так, как её «сосед»
    (защита от шрифтов, где глиф есть, но пустой или скопирован)."""
    font = ImageFont.truetype(path, 40)
    def bitmap(ch):
        im = Image.new("L", (60, 60), 0)
        ImageDraw.Draw(im).text((10, 0), ch, font=font, fill=255)
        return np.asarray(im) > 128
    for a, b in pairs:
        ba = bitmap(a)
        if ba.sum() < 20 or (b is not None and (ba == bitmap(b)).all()):
            return False
    return True


# ---------------------------------------------------------------- рендеринг
def render_line(text, font_path, size=None, rng=random):
    """Рендер строки на чистом фоне. Возвращает uint8 серое изображение."""
    size = size or rng.randint(26, 52)
    font = ImageFont.truetype(font_path, size)
    spacing = rng.choice([0, 0, 0, 0, 1, 2, -1]) * size / 30.0
    ink = rng.randint(0, 60)
    bg = rng.randint(215, 255)
    # ширина с учётом межбуквенного интервала
    if spacing == 0:
        w = font.getlength(text)
    else:
        w = sum(font.getlength(ch) for ch in text) + spacing * (len(text) - 1)
    asc, desc = font.getmetrics()
    pad_x = int(rng.uniform(0.1, 1.0) * size)
    pad_top = int(rng.uniform(0.05, 0.4) * size)
    pad_bot = int(rng.uniform(0.05, 0.4) * size)
    W = int(w) + 2 * pad_x
    H = asc + desc + pad_top + pad_bot
    im = Image.new("L", (W, H), bg)
    d = ImageDraw.Draw(im)
    if spacing == 0:
        d.text((pad_x, pad_top), text, font=font, fill=ink)
    else:
        x = pad_x
        for ch in text:
            d.text((x, pad_top), ch, font=font, fill=ink)
            x += font.getlength(ch) + spacing
    return np.asarray(im)


# ---------------------------------------------------------------- искажения
def _rotate(img, angle, bg):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=int(bg))


def _gauss_noise(img, sigma, rng):
    n = np.random.default_rng(rng.randint(0, 2**31)).normal(0, sigma, img.shape)
    return np.clip(img.astype(np.float32) + n, 0, 255).astype(np.uint8)


def _jpeg(img, q):
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, int(q)])
    return cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)


def _ink_thickness(img, rng):
    """Утолщение/утоньшение штрихов (перерасход или нехватка тонера)."""
    k = np.ones((2, 2), np.uint8)
    if rng.random() < 0.5:
        return cv2.erode(img, k)     # тёмный текст на светлом фоне -> толще
    return cv2.dilate(img, k)        # -> тоньше


def _bleed_through(img, rng):
    """Просвечивание текста с обратной стороны листа."""
    ghost = cv2.flip(img, 1)
    ghost = cv2.GaussianBlur(ghost, (0, 0), rng.uniform(1.0, 2.0))
    shift = rng.randint(-img.shape[0] // 4, img.shape[0] // 4)
    ghost = np.roll(ghost, shift, axis=0)
    a = rng.uniform(0.08, 0.25)
    # «призрак» слегка затемняет фон, но не осветляет основной текст
    ghost_layer = 255 - a * (255 - ghost.astype(np.float32))
    return np.minimum(img.astype(np.float32), ghost_layer).astype(np.uint8)


def _binarize_artifacts(img, rng):
    """Грубая бинаризация, как у дешёвых сканеров и факсов."""
    thr = rng.randint(110, 170)
    return np.where(img > thr, 255, 0).astype(np.uint8)


def _paper_texture(img, rng):
    h, w = img.shape
    small = np.random.default_rng(rng.randint(0, 2**31)).normal(
        0, 1, (max(2, h // 8), max(2, w // 8))).astype(np.float32)
    tex = cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC)
    return np.clip(img.astype(np.float32) + tex * rng.uniform(3, 10), 0, 255).astype(np.uint8)


def distort_scan(img, rng):
    bg = int(np.median(img[0]))
    if rng.random() < 0.8:
        img = _rotate(img, rng.uniform(-2, 2), bg)
    if rng.random() < 0.3:
        img = _ink_thickness(img, rng)
    if rng.random() < 0.25:
        img = _bleed_through(img, rng)
    if rng.random() < 0.5:
        img = _paper_texture(img, rng)
    if rng.random() < 0.6:
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(0.3, 1.2))
    if rng.random() < 0.15:
        img = _binarize_artifacts(img, rng)
    if rng.random() < 0.6:
        img = _gauss_noise(img, rng.uniform(2, 12), rng)
    if rng.random() < 0.6:
        img = _jpeg(img, rng.randint(25, 90))
    return img


# Режимы для экспериментов
MODES = ("clean", "scan")


def make_sample(text, font_path, mode="scan", height=32, max_width=512, rng=random):
    """Полный цикл: рендеринг -> искажения сканирования -> приведение к высоте height.
    mode="clean" — без искажений; mode="scan" — с искажениями (10 % строк остаются чистыми)."""
    assert mode in MODES, mode
    img = render_line(text, font_path, rng=rng)
    if mode == "scan" and rng.random() >= 0.1:
        img = distort_scan(img, rng)
    h, w = img.shape
    new_w = max(8, min(max_width, int(round(w * height / h))))
    return cv2.resize(img, (new_w, height), interpolation=cv2.INTER_AREA)


# ---------------------------------------------------------------- массовая генерация
_G = {}


def _init_worker(lines, weights, fonts, mode, height, max_width, seed):
    _G.update(lines=lines, cum=np.cumsum(weights), fonts=fonts, mode=mode,
              height=height, max_width=max_width, seed=seed)


def _make_one(args):
    idx, out_dir = args
    rng = random.Random(_G["seed"] * 1_000_003 + idx)
    j = int(np.searchsorted(_G["cum"], rng.random() * _G["cum"][-1]))
    text = _G["lines"][min(j, len(_G["lines"]) - 1)]
    font = rng.choice(_G["fonts"])
    try:
        img = make_sample(text, font, _G["mode"], _G["height"], _G["max_width"], rng)
    except Exception:
        return None
    name = f"{idx:07d}.png"
    cv2.imwrite(f"{out_dir}/{name}", img)
    return f"{name}\t{text}\t{font.split('/')[-1]}"


def generate_set(lines, fonts, mode, n, out_dir, seed=0, height=32,
                 max_width=512, weighted=True, workers=None, chunks=64):
    """Генерирует n изображений в out_dir и файл labels.tsv (имя, текст, шрифт)."""
    import os
    from multiprocessing import Pool
    os.makedirs(out_dir, exist_ok=True)
    weights = [tatar_weight(s) if weighted else 1.0 for s in lines]
    rows = []
    with Pool(workers or os.cpu_count(), _init_worker,
              (lines, weights, fonts, mode, height, max_width, seed)) as pool:
        for k, r in enumerate(pool.imap(_make_one, ((i, out_dir) for i in range(n)),
                                        chunksize=chunks)):
            if r:
                rows.append(r)
            if (k + 1) % 20000 == 0:
                print(f"  {out_dir}: {k + 1}/{n}")
    with open(f"{out_dir}/labels.tsv", "w", encoding="utf-8") as f:
        f.write("\n".join(rows) + "\n")
    return len(rows)


# ---------------------------------------------------------------- страницы для печати
def make_print_pages(lines, fonts, pt_sizes=(12, 14), dpi=300, lines_per_page=22,
                     rng=random):
    """Вёрстка страниц A4 для реального теста (печать -> сканирование).
    Строки разнесены с большим интервалом, чтобы потом их было легко нарезать.
    Возвращает (список PIL-страниц, список записей разметки)."""
    W, H = int(8.27 * dpi), int(11.69 * dpi)
    margin = int(0.8 * dpi)
    pages, meta, k = [], [], 0
    while k < len(lines):
        font_path = fonts[len(pages) % len(fonts)]
        pt = pt_sizes[len(pages) % len(pt_sizes)]
        px = int(pt / 72 * dpi)
        font = ImageFont.truetype(font_path, px)
        hfont = ImageFont.truetype(font_path, int(9 / 72 * dpi))
        page = Image.new("L", (W, H), 255)
        d = ImageDraw.Draw(page)
        d.text((margin, margin // 2), f"стр. {len(pages) + 1}", font=hfont, fill=0)
        step = (H - 2 * margin) // lines_per_page
        for r in range(lines_per_page):
            if k >= len(lines):
                break
            text = lines[k]
            while font.getlength(text) > W - 2 * margin:   # на всякий случай
                text = text[:-1]
            d.text((margin, margin + r * step), text, font=font, fill=0)
            meta.append({"page": len(pages) + 1, "row": r + 1, "text": text,
                         "font": font_path.split("/")[-1], "pt": pt})
            k += 1
        pages.append(page)
    return pages, meta

In [ ]:
import importlib, tatsynth as T
importlib.reload(T)
print('Символов в алфавите модели:', len(T.ALPHABET))
print(T.ALPHABET)
# проверка нормализации: латинские h, ə, Y и пр. внутри татарских слов
print(T.normalize('Hәр кешенең hәм yзе, Ӱрнәк ӓни ə-ɵ'))

## 3. Текстовый корпус

Источники: татарская Википедия (`wikimedia/wikipedia`, конфигурация `20231101.tt`) и `neurotatarlar/tt-books-cyrillic`.
Документы делятся на обучающие и тестовые **по документу** (хеш названия), чтобы тексты теста не встречались при обучении.
Строки с латиницей и чужими символами отбрасываются, строки с ә, ө, ү, җ, ң, һ при генерации выбираются чаще.

In [ ]:
import hashlib, random, collections
from datasets import load_dataset

MAX_ARTICLES = 80000        # статей Википедии (их ~500 тыс., но много однотипных заготовок)
MAX_BOOKS = 300             # книг
MAX_CHARS_PER_BOOK = 300_000
MAX_LINES_PER_DOC = 150     # чтобы одна длинная книга не заполнила весь корпус
TEST_SHARE = 0.03

def iter_docs():
    try:
        ds = load_dataset('wikimedia/wikipedia', '20231101.tt', split='train', streaming=True)
        for i, ex in enumerate(ds):
            if i >= MAX_ARTICLES: break
            yield 'wiki', ex['title'], ex['text']
    except Exception as e:
        print('Википедия не загрузилась:', e)
    try:
        ds = load_dataset('neurotatarlar/tt-books-cyrillic', split='train', streaming=True)
        for i, ex in enumerate(ds):
            if i >= MAX_BOOKS: break
            txt = max((v for v in ex.values() if isinstance(v, str)), key=len, default='')
            if len(txt) > 500:
                yield 'books', f'book{i}', txt[:MAX_CHARS_PER_BOOK]
    except Exception as e:
        print('Книги не загрузились:', e)

lines_path = {s: f'{ROOT}/{s}_lines.txt' for s in ('train', 'test')}
if all(os.path.exists(p) for p in lines_path.values()):
    print('Корпус уже собран, читаю с Диска')
else:
    rng = random.Random(0)
    out = {'train': set(), 'test': set()}
    stat = collections.Counter()
    for src, title, text in iter_docs():
        split = 'test' if int(hashlib.md5(title.encode()).hexdigest(), 16) % 1000 < TEST_SHARE * 1000 else 'train'
        doc_lines = []
        for par in text.split('\n'):
            par = T.normalize(par)
            if len(par) < 20 or not T.looks_tatar(par):
                continue
            doc_lines += T.text_to_lines(par, rng=rng)
        rng.shuffle(doc_lines)
        out[split].update(doc_lines[:MAX_LINES_PER_DOC])
        stat[src] += 1
        if sum(stat.values()) % 10000 == 0:
            print(dict(stat), {k: len(v) for k, v in out.items()})
    out['test'] -= out['train']                       # на всякий случай убираем пересечения
    for s in out:
        with open(lines_path[s], 'w', encoding='utf-8') as f:
            f.write('\n'.join(sorted(out[s])))
    print('Документов:', dict(stat))

train_lines = open(lines_path['train'], encoding='utf-8').read().splitlines()
test_lines = open(lines_path['test'], encoding='utf-8').read().splitlines()
print(f'Строк: обучение {len(train_lines)}, тест {len(test_lines)}')

In [ ]:
# Статистика корпуса (пригодится для раздела статьи про корпус)
def corpus_stats(lines):
    if not lines:
        print('  пусто'); return
    chars = collections.Counter(ch.lower() for s in lines for ch in s)
    total = sum(chars.values())
    with_tt = sum(any(ch in T.TT_SPECIAL for ch in s) for s in lines)
    print(f'  строк: {len(lines)}, символов: {total}, средняя длина: {total/len(lines):.1f}')
    print(f'  строк со специфическими буквами: {with_tt/len(lines):.1%}')
    for ch in T.TT_LOWER:
        print(f'    {ch}: {chars[ch]/total:.2%}')
print('Обучение:'); corpus_stats(train_lines)
print('Тест:'); corpus_stats(test_lines)
print('Примеры:', *random.sample(train_lines, 8), sep='\n  ')

## 4. Шрифты

Шрифт подходит, если в нём есть все символы алфавита и татарские буквы нарисованы отдельно от «соседей» (ә≠е, ө≠о, ү≠у, җ≠ж, ң≠н).
Часть семейств шрифтов откладываем: они не используются при обучении и нужны, чтобы проверить работу модели на незнакомых шрифтах.

In [ ]:
import glob, json
from fontTools.ttLib import TTFont

EXCLUDE = ('unifont', 'emoji', 'math', 'symbol', 'sample', 'dingbat', 'icon')
# Обычные «документные» шрифты: по два из них идут в обучение и в отложенные,
# и именно ими верстается PDF для реального теста
DOC_FONTS = ['Liberation Serif', 'PT Serif', 'Liberation Sans', 'PT Sans', 'DejaVu Serif',
             'Noto Serif', 'Noto Sans', 'Carlito', 'Open Sans', 'Roboto', 'IBM Plex Serif', 'Ubuntu']

def font_names(path):
    try:
        n = TTFont(path, lazy=True)['name']
        fam = n.getDebugName(16) or n.getDebugName(1)     # 16 = общее семейство (без веса)
        style = n.getDebugName(17) or n.getDebugName(2)
        return str(fam), str(style)
    except Exception:
        return None, None

paths = sorted(set(glob.glob('/usr/share/fonts/**/*.[ot]tf', recursive=True)))
good, rejected = [], collections.Counter()
for p in paths:
    if any(x in p.lower() for x in EXCLUDE):
        rejected['служебный шрифт'] += 1; continue
    ok, missing = T.font_covers(p)
    if not ok:
        rejected['нет символов'] += 1; continue
    if not T.glyphs_distinct(p):
        rejected['татарские буквы не отличаются'] += 1; continue
    fam, style = font_names(p)
    if fam and not any(x in fam.lower() for x in EXCLUDE):
        good.append({'path': p, 'family': fam, 'style': style})
print(f'Проверено шрифтов: {len(paths)}, подходят: {len(good)}, отброшены: {dict(rejected)}')

families = sorted({g['family'] for g in good})
rng = random.Random(42)
doc = [f for f in DOC_FONTS if f in families]
rng.shuffle(doc)
holdout_fams = set(doc[:2])                               # два документных шрифта откладываем
others = [f for f in families if f not in doc]
holdout_fams |= set(rng.sample(others, max(0, len(families) // 5 - 2)))
train_fonts = [g['path'] for g in good if g['family'] not in holdout_fams]
holdout_fonts = [g['path'] for g in good if g['family'] in holdout_fams]
print(f'Семейств: {len(families)}, начертаний: {len(good)}')
print('Для обучения:', sorted(set(families) - holdout_fams))
print('Отложены (незнакомые шрифты):', sorted(holdout_fams))
json.dump({'train': train_fonts, 'holdout': holdout_fonts, 'all': good},
          open(f'{ROOT}/fonts.json', 'w'), ensure_ascii=False, indent=1)

In [ ]:
# Как каждое семейство рисует татарские буквы (рисунок можно взять в статью)
import matplotlib.pyplot as plt
fams = [next(g for g in good if g['family'] == f and g['style'] == 'Regular') if any(g['family'] == f and g['style'] == 'Regular' for g in good)
        else next(g for g in good if g['family'] == f) for f in families]
fig, axes = plt.subplots(len(fams), 1, figsize=(9, 0.55 * len(fams)))
for ax, g in zip(axes, fams):
    img = T.render_line('Әә Өө Үү Җҗ Ңң Һһ  Ее Оо Уу Жж Нн', g['path'], size=40, rng=random.Random(0))
    ax.imshow(img, cmap='gray', vmin=0, vmax=255); ax.set_axis_off()
    ax.set_title(g['family'] + (' (отложен)' if g['family'] in holdout_fams else ''), fontsize=8, loc='left')
plt.tight_layout(); plt.show()

## 5. Примеры синтетических изображений: без искажений и с искажениями сканирования

In [ ]:
rng = random.Random(7)
fig, axes = plt.subplots(2, 4, figsize=(16, 2.4))
for r, mode in enumerate(['clean', 'scan']):
    for c in range(4):
        s = rng.choice([l for l in train_lines[:20000] if any(ch in T.TT_SPECIAL for ch in l)])
        img = T.make_sample(s, rng.choice(train_fonts), mode=mode, height=48, rng=rng)
        axes[r, c].imshow(img, cmap='gray', vmin=0, vmax=255); axes[r, c].set_axis_off()
        axes[r, c].set_title(f'{mode}: {s}', fontsize=7, loc='left')
plt.tight_layout()
plt.savefig(f'{ROOT}/fig_examples.jpg', dpi=200)    # рисунок для статьи (JPEG по правилам)
plt.show()

## 6. Генерация наборов данных

Если вы уже запускали прошлую версию блокнота, файлы `train_full.tar`, `train_photo.tar`, `train_clean.tar`, `train_scan.tar`, `val_synth.tar`, `test_synth_unseen.tar` в папке `data` можно удалить: новые наборы называются иначе.

* `train_clean_*`, `train_scan_*` — обучающие наборы одинакового размера: без искажений и с искажениями сканирования (их сравнение показывает, что дают искажения);
* `val_synth` — валидация: тестовые тексты, обучающие шрифты;
* `test_synth_unseen` — тестовые тексты, **отложенные** шрифты.

Картинки высотой 32 пикселя. Каждый набор пакуется в один .tar на Диск (сотни тысяч мелких файлов на Диск лучше не писать).
На бесплатном Colab 50 тыс. изображений генерируются примерно за 5–7 минут, весь раздел — минут за 15, на Диске около 0,7 ГБ.

In [ ]:
import time, subprocess
N_TRAIN = 50_000
N_EVAL = 5_000
k = f'{N_TRAIN // 1000}k'
SETS = {
    #  имя                     режим    число     строки        шрифты         seed
    f'train_clean_{k}':       ('clean', N_TRAIN, train_lines, train_fonts,   2),
    f'train_scan_{k}':        ('scan',  N_TRAIN, train_lines, train_fonts,   3),
    'val_scan':               ('scan',  N_EVAL,  test_lines,  train_fonts,   5),
    'test_scan_unseen_fonts': ('scan',  N_EVAL,  test_lines,  holdout_fonts, 6),
}
for name, (mode, n, lines, fonts, seed) in SETS.items():
    tar = f'{ROOT}/data/{name}.tar'
    if os.path.exists(tar):
        print(f'{name}: уже есть на Диске'); continue
    t = time.time()
    cnt = T.generate_set(lines, fonts, mode, n, f'{LOCAL}/{name}', seed=seed, height=32)
    subprocess.run(['tar', '-cf', tar, '-C', LOCAL, name], check=True)
    print(f'{name}: {cnt} изображений, {time.time() - t:.0f} с')
json.dump({'alphabet': T.ALPHABET, 'height': 32, 'sets': list(SETS)}, open(f'{ROOT}/data/meta.json', 'w'), ensure_ascii=False)

## 7. PDF для реального теста

Из **тестовых** текстов отбираются строки (в основном со специфическими татарскими буквами) и верстаются на страницы A4 четырьмя шрифтами: два знакомых модели и два отложенных.

**Что сделать после этой ячейки:**
1. скачать `print_test/print_test.pdf` с Диска и распечатать;
2. отсканировать все страницы: 300 dpi, оттенки серого, формат JPEG или PNG; лист класть ровно, но идеальная точность не нужна;
3. назвать файлы по номеру страницы: `scan_01.jpg`, `scan_02.jpg`, … (номер напечатан вверху каждой страницы);
4. загрузить файлы в папку `tatar_ocr/print_test/scan/` на Диске.

Нарезку страниц на строки сделаем в блокноте 3.

In [ ]:
N_PRINT_LINES = 220
rng = random.Random(11)
pool_tt = [l for l in test_lines if 20 <= len(l) <= 48 and any(ch in T.TT_SPECIAL for ch in l)]
pool_ru = [l for l in test_lines if 20 <= len(l) <= 48 and not any(ch in T.TT_SPECIAL for ch in l)]
n_tt = min(len(pool_tt), int(N_PRINT_LINES * 0.8))
chosen = rng.sample(pool_tt, n_tt) + rng.sample(pool_ru, min(len(pool_ru), N_PRINT_LINES - n_tt))
rng.shuffle(chosen)

def pick_doc(fonts, k):
    # k обычных (Regular) начертаний документных шрифтов из списка fonts
    cand = {}
    for g in good:
        if g['path'] in fonts and g['family'] in DOC_FONTS and g['style'] in ('Regular', 'Book', 'Normal', 'Roman'):
            cand.setdefault(g['family'], g['path'])
    picked = list(cand.values())[:k]
    return picked + rng.sample(fonts, k - len(picked)) if len(picked) < k else picked
print_fonts = pick_doc(train_fonts, 2) + pick_doc(holdout_fonts, 2)
print('Шрифты для печати:', [p.split('/')[-1] for p in print_fonts])

pages, meta = T.make_print_pages(chosen, print_fonts, pt_sizes=(12, 14))
for m in meta:
    m['seen_font'] = any(m['font'] == p.split('/')[-1] for p in train_fonts)
pages[0].save(f'{ROOT}/print_test/print_test.pdf', save_all=True, append_images=pages[1:], resolution=300)
json.dump(meta, open(f'{ROOT}/print_test/labels.json', 'w'), ensure_ascii=False, indent=1)
os.makedirs(f'{ROOT}/print_test/scan', exist_ok=True)
print(f'Страниц: {len(pages)}, строк: {len(meta)}')
plt.figure(figsize=(6, 8)); plt.imshow(pages[0], cmap='gray'); plt.axis('off'); plt.show()

## Что получилось

На Диске в `tatar_ocr/`:
* `train_lines.txt`, `test_lines.txt` — корпус;
* `fonts.json` — шрифты (обучающие и отложенные);
* `data/*.tar` — синтетические наборы, `data/meta.json` — алфавит;
* `fig_examples.jpg` — рисунок с примерами для статьи;
* `print_test/` — PDF для печати, разметка к нему и папка `scan/` для сканов.

Дальше: блокнот 2 (обучение CRNN на GPU).